In [44]:
import pandas as pd
import numpy as np
import tensorflow as tf
from tensorflow.keras.layers import Input, Embedding, Dense, Dropout, Conv1D, MaxPooling1D, Bidirectional, LSTM, Flatten, Concatenate
from tensorflow.keras.models import Model
from sklearn.preprocessing import LabelEncoder, StandardScaler
from sklearn.model_selection import train_test_split


In [45]:
train = pd.read_csv(r"C:\Users\Mohammed Yaseer\Desktop\CarPriceProject\data\train.csv")
test = pd.read_csv(r"C:\Users\Mohammed Yaseer\Desktop\CarPriceProject\data\test.csv")

train.head()


,ID,Price,Levy,Manufacturer,Model,Prod. year,Category,Leather interior,Fuel type,Engine volume,...,Drive wheels,Doors,Wheel,Color,Airbags,Accident_history,Brand_value_index,Safety_features,Comfort_features,Modification_penalty
0,45654403,13328,1399,LEXUS,RX 450,2010,Jeep,Yes,Hybrid,3.5,...,4x4,4-May,Left wheel,Silver,12,0,3,1,10,1
1,44731507,16621,1018,CHEVROLET,Equinox,2011,Jeep,No,Petrol,3,...,4x4,4-May,Left wheel,Black,8,0,7,9,2,1
2,45774419,8467,-,HONDA,FIT,2006,Hatchback,No,Petrol,1.3,...,Front,4-May,Right-hand drive,Black,2,0,8,5,10,1
3,45769185,3607,862,FORD,Escape,2011,Jeep,Yes,Hybrid,2.5,...,4x4,4-May,Left wheel,White,0,0,2,8,1,0
4,45809263,11726,446,HONDA,FIT,2014,Hatchback,Yes,Petrol,1.3,...,Front,4-May,Left wheel,Silver,4,1,5,4,10,0


In [46]:
y = train["Price"]
X = train.drop(["Price", "ID"], axis=1)

if "Price" in test.columns:
    test = test.drop("Price", axis=1)
if "ID" in test.columns:
    test = test.drop("ID", axis=1)

In [47]:
with open("scaler.pkl", "wb") as f:
    pickle.dump(scaler, f)

with open("encoders.pkl", "wb") as f:
    pickle.dump(label_encoders, f)

In [48]:
categorical_cols = X.select_dtypes(include=['object']).columns.tolist()
numeric_cols = X.select_dtypes(include=['int64', 'float64']).columns.tolist()

categorical_cols, numeric_cols


(['Levy',
  'Manufacturer',
  'Model',
  'Category',
  'Leather interior',
  'Fuel type',
  'Engine volume',
  'Mileage',
  'Gear box type',
  'Drive wheels',
  'Doors',
  'Wheel',
  'Color'],
 ['Prod. year',
  'Cylinders',
  'Airbags',
  'Accident_history',
  'Brand_value_index',
  'Safety_features',
  'Comfort_features',
  'Modification_penalty'])

In [49]:
from sklearn.preprocessing import LabelEncoder

# Identify categorical columns (non-numeric)
cat_cols = X.select_dtypes(include=['object']).columns.tolist()

label_encoders = {}

# Safe encoding for unseen labels in test set
for col in cat_cols:
    le = LabelEncoder()

    # Combine train & test values → prevents unseen label errors
    combined = pd.concat([X[col].astype(str), test[col].astype(str)], axis=0)

    le.fit(combined)

    # Transform both dataset
    X[col] = le.transform(X[col].astype(str))
    test[col] = le.transform(test[col].astype(str))

    label_encoders[col] = le

print("Safe encoding completed for:", cat_cols)


Safe encoding completed for: ['Levy', 'Manufacturer', 'Model', 'Category', 'Leather interior', 'Fuel type', 'Engine volume', 'Mileage', 'Gear box type', 'Drive wheels', 'Doors', 'Wheel', 'Color']


In [50]:
# ---------------------------
# UPDATED LABEL ENCODING CELL
# ---------------------------

from sklearn.preprocessing import LabelEncoder

# Identify categorical columns (object or string type)
cat_cols = X.select_dtypes(include=['object']).columns.tolist()

label_encoders = {}

# Encode categorical columns safely (handles unseen labels)
for col in cat_cols:
    le = LabelEncoder()

    # Combine train + test values so no label is unseen
    combined = pd.concat([X[col].astype(str), test[col].astype(str)], axis=0)

    le.fit(combined)

    # Transform train and test
    X[col] = le.transform(X[col].astype(str))
    test[col] = le.transform(test[col].astype(str))

    label_encoders[col] = le

print("Label encoding completed for columns:", cat_cols)

# ----------------------------------------
# SAVE LABEL ENCODERS   <<<<<< ADD THIS
# ----------------------------------------
import pickle, os

os.makedirs("model", exist_ok=True)

with open("model/encoders.pkl", "wb") as f:
    pickle.dump(label_encoders, f)

print("Encoders saved successfully to model/encoders.pkl")



Label encoding completed for columns: []
Encoders saved successfully to model/encoders.pkl


In [51]:
from sklearn.preprocessing import StandardScaler

# Remove Price column if present in test
if "Price" in test.columns:
    test = test.drop("Price", axis=1)

scaler = StandardScaler()

X_scaled = scaler.fit_transform(X)
test_scaled = scaler.transform(test)

print("Scaling completed successfully.")


Scaling completed successfully.


In [52]:
from sklearn.model_selection import train_test_split

X_train, X_val, y_train, y_val = train_test_split(
    X_scaled, y, test_size=0.2, random_state=42
)

X_train.shape, X_val.shape


((15389, 21), (3848, 21))

In [53]:
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense, Conv1D, MaxPooling1D, Flatten, Bidirectional, LSTM, Dropout, Reshape

n_features = X_train.shape[1]

model = Sequential([
    Reshape((n_features, 1), input_shape=(n_features,)),

    # CNN Block
    Conv1D(64, kernel_size=3, activation='relu'),
    MaxPooling1D(pool_size=2),

    # BiLSTM Block
    Bidirectional(LSTM(64, return_sequences=False)),
    Dropout(0.3),

    # Dense Layers
    Dense(64, activation='relu'),
    Dropout(0.2),

    Dense(1)  # Price output
])

model.compile(optimizer='adam', loss='mse', metrics=['mae'])
model.summary()


C:\ProgramData\anaconda3\Lib\site-packages\keras\src\layers\reshaping\reshape.py:38: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(**kwargs)


Model: "sequential_3"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ reshape_3 (Reshape)             │ (None, 21, 1)          │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv1d_3 (Conv1D)               │ (None, 19, 64)         │           256 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling1d_3 (MaxPooling1D)  │ (None, 9, 64)          │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ bidirectional_3 (Bidirectional) │ (None, 128)            │        66,048 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_6 (Dropout)             │ (None, 128)            │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_6 (Dense)                 │ (None, 64)             │         8,256 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_7 (Dropout)             │ (None, 64)             │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_7 (Dense)                 │ (None, 1)              │            65 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 74,625 (291.50 KB)

 Trainable params: 74,625 (291.50 KB)

 Non-trainable params: 0 (0.00 B)

In [54]:
history = model.fit(
    X_train, y_train,
    epochs=25,
    batch_size=32,
    validation_split=0.2,
    verbose=1
)


Epoch 1/25
385/385 ━━━━━━━━━━━━━━━━━━━━ 21s 24ms/step - loss: 56932601856.0000 - mae: 19048.7480 - val_loss: 550254080.0000 - val_mae: 15628.3066
Epoch 2/25
385/385 ━━━━━━━━━━━━━━━━━━━━ 8s 20ms/step - loss: 56886321152.0000 - mae: 18051.3770 - val_loss: 515150272.0000 - val_mae: 14835.7773
Epoch 3/25
385/385 ━━━━━━━━━━━━━━━━━━━━ 8s 20ms/step - loss: 56841154560.0000 - mae: 17215.5586 - val_loss: 478726464.0000 - val_mae: 14017.7578
Epoch 4/25
385/385 ━━━━━━━━━━━━━━━━━━━━ 7s 19ms/step - loss: 56794542080.0000 - mae: 16423.2148 - val_loss: 440175008.0000 - val_mae: 13162.5195
Epoch 5/25
385/385 ━━━━━━━━━━━━━━━━━━━━ 8s 20ms/step - loss: 56751464448.0000 - mae: 15644.0244 - val_loss: 408681952.0000 - val_mae: 12478.3418
Epoch 6/25
385/385 ━━━━━━━━━━━━━━━━━━━━ 8s 20ms/step - loss: 56707612672.0000 - mae: 15044.7344 - val_loss: 377528000.0000 - val_mae: 11870.8066
Epoch 7/25
385/385 ━━━━━━━━━━━━━━━━━━━━ 8s 20ms/step - loss: 56670867456.0000 - mae: 14527.4688 - val_loss: 354672864.0000 - val_

In [55]:
test_predictions = model.predict(test_scaled)
test_predictions[:10]


258/258 ━━━━━━━━━━━━━━━━━━━━ 4s 12ms/step


array([[17774.686],
       [21941.203],
       [14819.757],
       [13517.641],
       [12472.32 ],
       [21246.744],
       [19927.346],
       [11176.721],
       [17476.809],
       [12045.559]], dtype=float32)

In [56]:
import pickle

# Save model
model.save("car_price_model.h5")

# Save scaler
with open("scaler.pkl", "wb") as f:
    pickle.dump(scaler, f)

# Save label encoders
with open("encoders.pkl", "wb") as f:
    pickle.dump(label_encoders, f)

print("All files saved successfully.")


All files saved successfully.


In [41]:
import pickle
scaler = pickle.load(open("./model/scaler.pkl", "rb"))
print(scaler.feature_names_in_)

['Levy' 'Manufacturer' 'Model' 'Prod. year' 'Category' 'Leather interior'
 'Fuel type' 'Engine volume' 'Mileage' 'Cylinders' 'Gear box type'
 'Drive wheels' 'Doors' 'Wheel' 'Color' 'Airbags' 'Accident_history'
 'Brand_value_index' 'Safety_features' 'Comfort_features'
 'Modification_penalty']


In [57]:
import pandas as pd
import numpy as np
import tensorflow as tf
from sklearn.preprocessing import LabelEncoder, StandardScaler
from sklearn.model_selection import train_test_split
import pickle
import os

# ------------------------------
# LOAD DATA
# ------------------------------
train = pd.read_csv(r"C:\Users\Mohammed Yaseer\Desktop\CarPriceProject\data\train.csv")
test = pd.read_csv(r"C:\Users\Mohammed Yaseer\Desktop\CarPriceProject\data\test.csv")

y = train["Price"]
X = train.drop(["Price", "ID"], axis=1)

if "Price" in test.columns:
    test = test.drop("Price", axis=1)
if "ID" in test.columns:
    test = test.drop("ID", axis=1)

# ------------------------------
# IDENTIFY CATEGORICAL COLUMNS
# ------------------------------
cat_cols = X.select_dtypes(include=['object']).columns.tolist()

label_encoders = {}

# ------------------------------
# SAFE LABEL ENCODING
# ------------------------------
for col in cat_cols:
    le = LabelEncoder()

    # Combine train + test so streamlit won't get unseen label errors
    combined = pd.concat([X[col].astype(str), test[col].astype(str)], axis=0)

    le.fit(combined)

    X[col] = le.transform(X[col].astype(str))
    test[col] = le.transform(test[col].astype(str))

    label_encoders[col] = le

print("\nLabel Encoding Completed For:", cat_cols)

# ------------------------------
# SAVE ENCODERS
# ------------------------------
os.makedirs("model", exist_ok=True)

with open("model/encoders.pkl", "wb") as f:
    pickle.dump(label_encoders, f)

print("Encoders saved → model/encoders.pkl")

# ------------------------------
# SCALE NUMERICAL COLUMNS
# ------------------------------
scaler = StandardScaler()

X_scaled = scaler.fit_transform(X)
test_scaled = scaler.transform(test)

with open("model/scaler.pkl", "wb") as f:
    pickle.dump(scaler, f)

print("Scaler saved → model/scaler.pkl")

# ------------------------------
# TRAIN–TEST SPLIT
# ------------------------------
X_train, X_val, y_train, y_val = train_test_split(
    X_scaled, y, test_size=0.2, random_state=42
)

n_features = X_train.shape[1]

# ------------------------------
# BUILD MODEL
# ------------------------------
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense, Conv1D, MaxPooling1D, Flatten, Bidirectional, LSTM, Dropout, Reshape

model = Sequential([
    Reshape((n_features, 1), input_shape=(n_features,)),

    Conv1D(64, kernel_size=3, activation='relu'),
    MaxPooling1D(pool_size=2),

    Bidirectional(LSTM(64, return_sequences=False)),
    Dropout(0.3),

    Dense(64, activation='relu'),
    Dropout(0.2),

    Dense(1)
])

model.compile(optimizer='adam', loss='mse', metrics=['mae'])
model.summary()

# ------------------------------
# TRAIN
# ------------------------------
model.fit(
    X_train, y_train,
    epochs=25,
    batch_size=32,
    validation_data=(X_val, y_val),
    verbose=1
)

# ------------------------------
# SAVE MODEL
# ------------------------------
model.save("model/car_price_model.h5")

print("\nModel saved → model/car_price_model.h5")
print("\n🎉 Training Pipeline Complete!")



Label Encoding Completed For: ['Levy', 'Manufacturer', 'Model', 'Category', 'Leather interior', 'Fuel type', 'Engine volume', 'Mileage', 'Gear box type', 'Drive wheels', 'Doors', 'Wheel', 'Color']
Encoders saved → model/encoders.pkl
Scaler saved → model/scaler.pkl


C:\ProgramData\anaconda3\Lib\site-packages\keras\src\layers\reshaping\reshape.py:38: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(**kwargs)


Model: "sequential_4"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ reshape_4 (Reshape)             │ (None, 21, 1)          │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv1d_4 (Conv1D)               │ (None, 19, 64)         │           256 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling1d_4 (MaxPooling1D)  │ (None, 9, 64)          │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ bidirectional_4 (Bidirectional) │ (None, 128)            │        66,048 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_8 (Dropout)             │ (None, 128)            │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_8 (Dense)                 │ (None, 64)             │         8,256 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_9 (Dropout)             │ (None, 64)             │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_9 (Dense)                 │ (None, 1)              │            65 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 74,625 (291.50 KB)

 Trainable params: 74,625 (291.50 KB)

 Non-trainable params: 0 (0.00 B)

Epoch 1/25
481/481 ━━━━━━━━━━━━━━━━━━━━ 26s 26ms/step - loss: 45659881472.0000 - mae: 18432.6855 - val_loss: 554891520.0000 - val_mae: 15853.0811
Epoch 2/25
481/481 ━━━━━━━━━━━━━━━━━━━━ 18s 21ms/step - loss: 45587697664.0000 - mae: 16986.7910 - val_loss: 483429600.0000 - val_mae: 14235.7637
Epoch 3/25
481/481 ━━━━━━━━━━━━━━━━━━━━ 10s 21ms/step - loss: 45515177984.0000 - mae: 15609.3643 - val_loss: 426192032.0000 - val_mae: 12989.8936
Epoch 4/25
481/481 ━━━━━━━━━━━━━━━━━━━━ 10s 21ms/step - loss: 45464592384.0000 - mae: 14650.9229 - val_loss: 391593856.0000 - val_mae: 12281.7803
Epoch 5/25
481/481 ━━━━━━━━━━━━━━━━━━━━ 11s 22ms/step - loss: 45428346880.0000 - mae: 14113.8955 - val_loss: 361967296.0000 - val_mae: 11772.2158
Epoch 6/25
481/481 ━━━━━━━━━━━━━━━━━━━━ 11s 22ms/step - loss: 45397217280.0000 - mae: 13719.6367 - val_loss: 342086656.0000 - val_mae: 11520.5234
Epoch 7/25
481/481 ━━━━━━━━━━━━━━━━━━━━ 10s 21ms/step - loss: 45367631872.0000 - mae: 13564.0244 - val_loss: 330365472.0000 


Model saved → model/car_price_model.h5

🎉 Training Pipeline Complete!


In [62]:
# model_training_dl.py
import os
import re
import pickle
import numpy as np
import pandas as pd
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.model_selection import train_test_split
import tensorflow as tf
from tensorflow.keras.layers import Input, Embedding, Dense, Dropout, Concatenate, Reshape, Conv1D, MaxPooling1D, Bidirectional, LSTM, Flatten
from tensorflow.keras.models import Model
from tensorflow.keras.callbacks import EarlyStopping, ModelCheckpoint, ReduceLROnPlateau
import re

def safe_name(col):
    # Replace all characters except letters, numbers, and underscores with "_"
    return re.sub(r'[^A-Za-z0-9_]', '_', col)


# -----------------------
# CONFIG
# -----------------------
CSV_PATH = r"C:\Users\Mohammed Yaseer\Desktop\CarPriceProject\data\train.csv"   # change path if needed
OUT_DIR = "model"
os.makedirs(OUT_DIR, exist_ok=True)

TARGET_COL = "Price"
LOG_TARGET = True           # Q1 choice: train on log(Price)
EPOCHS = 50                 # Q2 balanced
BATCH_SIZE = 64
VALIDATION_SPLIT = 0.2
RANDOM_STATE = 42

# -----------------------
# HELPERS
# -----------------------
def to_numeric_from_string(s):
    if pd.isna(s):
        return np.nan
    s = str(s).strip()
    s = re.sub(r'[^\d\.]', '', s)
    if s == "":
        return np.nan
    try:
        return float(s)
    except:
        return np.nan

def safe_label_encode(series):
    le = LabelEncoder()
    filled = series.fillna("___NA___").astype(str)
    le.fit(filled)
    return le, le.transform(filled), le.classes_

# -----------------------
# LOAD DATA
# -----------------------
df = pd.read_csv(CSV_PATH)
print("Loaded:", df.shape)

# Basic cleaning & numeric columns
for col in ["Levy", "Engine volume", "Mileage"]:
    if col in df.columns:
        df[col] = df[col].apply(to_numeric_from_string)

# Binary mappings
for col in ["Leather interior", "Accident_history"]:
    if col in df.columns:
        df[col] = df[col].astype(str).str.strip().str.lower().map({"yes":1, "no":0, "true":1, "false":0})
        df[col] = df[col].fillna(0).astype(int)

# Prod. year numeric
if "Prod. year" in df.columns:
    df["Prod. year"] = pd.to_numeric(df["Prod. year"], errors="coerce").fillna(df["Prod. year"].median())

# Drop ID if present
if "ID" in df.columns:
    df = df.drop("ID", axis=1)

# Target
y = df[TARGET_COL].astype(float).values
if LOG_TARGET:
    y_train_target = np.log1p(y)
else:
    y_train_target = y

# Features
X = df.drop(TARGET_COL, axis=1)

# Identify categorical and numeric columns
cat_cols = X.select_dtypes(include=['object']).columns.tolist()
num_cols = X.select_dtypes(include=[np.number]).columns.tolist()

print("Categorical columns:", cat_cols)
print("Numeric columns:", num_cols)

# -----------------------
# ENCODERS FOR CATEGORICALS (LabelEncoder -> integer indices)
# -----------------------
encoders = {}
cat_input_dims = {}     # vocabulary sizes
cat_encoded_arrays = {}

for col in cat_cols:
    le = LabelEncoder()
    series = X[col].fillna("___NA___").astype(str)
    le.fit(series)
    encoders[col] = le
    cat_input_dims[col] = len(le.classes_) + 1   # +1 safety
    cat_encoded_arrays[col] = le.transform(series)

# Replace object cols with integer-encoded columns in X_encoded
X_encoded = X.copy()
for col in cat_cols:
    X_encoded[col] = cat_encoded_arrays[col]

# -----------------------
# SCALE NUMERIC COLUMNS
# -----------------------
scaler = StandardScaler()
X_encoded[num_cols] = scaler.fit_transform(X_encoded[num_cols])

# Save encoders & scaler (for inference)
with open(os.path.join(OUT_DIR, "encoders.pkl"), "wb") as f:
    pickle.dump(encoders, f)
with open(os.path.join(OUT_DIR, "scaler.pkl"), "wb") as f:
    pickle.dump(scaler, f)

# Save metadata
metadata = {"cat_cols": cat_cols, "num_cols": num_cols, "cat_input_dims": cat_input_dims, "log_target": LOG_TARGET}
with open(os.path.join(OUT_DIR, "metadata.pkl"), "wb") as f:
    pickle.dump(metadata, f)

# -----------------------
# BUILD KERAS MODEL (Functional API)
# -----------------------
cat_inputs = []
cat_embeddings = []

for col in cat_cols:
    vocab_size = cat_input_dims[col]
    emb_dim = min(50, max(4, int(1.6 * np.sqrt(max(2, vocab_size)))))

    # convert column name to safe layer name
    safe = safe_name(col)

    input_i = Input(shape=(1,), name=f"in_{safe}")
    emb = Embedding(input_dim=vocab_size, output_dim=emb_dim, name=f"emb_{safe}")(input_i)
    emb = Reshape((emb_dim,))(emb)

    cat_inputs.append(input_i)
    cat_embeddings.append(emb)

# Numeric input
num_input = Input(shape=(len(num_cols),), name="numeric_input")
# Project numeric vector to same embedding dim so we can create a sequence
proj_dim = max(8, EMBEDDING_DIM_BASE)
num_proj = Dense(proj_dim, activation="relu", name="num_proj")(num_input)

# ---------------------------
# FIXED TOKEN CREATION BLOCK (USE THIS)
# ---------------------------

# Determine common projection dimension for all tokens
embed_dims = [int(emb.shape[-1]) for emb in cat_embeddings]
COMMON_DIM = max(proj_dim, max(embed_dims))

from tensorflow.keras.layers import Dense, TimeDistributed

projected_tokens = []

# Project each categorical embedding to COMMON_DIM, then reshape
for emb in cat_embeddings:
    proj = Dense(COMMON_DIM, activation="relu")(emb)        # (None, COMMON_DIM)
    token = Reshape((1, COMMON_DIM))(proj)                  # (None, 1, COMMON_DIM)
    projected_tokens.append(token)

# Project numeric vector to COMMON_DIM, then reshape
num_proj_common = Dense(COMMON_DIM, activation="relu")(num_proj)
num_token = Reshape((1, COMMON_DIM))(num_proj_common)
projected_tokens.append(num_token)

# Concatenate all tokens into a sequence
sequence = Concatenate(axis=1)(projected_tokens)            # (None, seq_len, COMMON_DIM)

# Optional additional projection on sequence
sequence_proj = TimeDistributed(Dense(COMMON_DIM, activation="relu"))(sequence)


# CNN + Pooling
x = Conv1D(filters=64, kernel_size=2, activation="relu", padding="same")(sequence_proj)
x = MaxPooling1D(pool_size=1)(x)

# BiLSTM
x = Bidirectional(LSTM(64, return_sequences=False))(x)
x = Dropout(0.3)(x)

# Dense head
x = Dense(128, activation="relu")(x)
x = Dropout(0.25)(x)
x = Dense(64, activation="relu")(x)
x = Dropout(0.15)(x)
output = Dense(1, activation="linear", name="price_out")(x)

# Build model inputs list: all cat inputs + numeric input
inputs = cat_inputs + [num_input]
model = Model(inputs=inputs, outputs=output)
model.compile(optimizer=tf.keras.optimizers.Adam(learning_rate=1e-3), loss="mse", metrics=["mae"])
model.summary()

# -----------------------
# PREPARE MODEL INPUTS (dict) FROM X_encoded
# -----------------------
# Each categorical input is a single integer column (shape (n,1)), numeric input is the scaled numeric matrix
model_inputs = {}
for col in cat_cols:
    model_inputs[f"in_{safe_name(col)}"] = X_encoded[col].values.reshape(-1,1)
model_inputs["numeric_input"] = X_encoded[num_cols].astype(np.float32).values

# Train / val split
train_idx, val_idx = train_test_split(np.arange(X_encoded.shape[0]), test_size=VALIDATION_SPLIT, random_state=RANDOM_STATE)
train_inputs = {k: v[train_idx] for k, v in model_inputs.items()}
val_inputs = {k: v[val_idx] for k, v in model_inputs.items()}
y_tr = y_train_target[train_idx]
y_val = y_train_target[val_idx]

# -----------------------
# CALLBACKS
# -----------------------
checkpoint_path = os.path.join(OUT_DIR, "dl_car_price_model_best.h5")
callbacks = [
    EarlyStopping(monitor="val_loss", patience=8, restore_best_weights=True),
    ModelCheckpoint(checkpoint_path, monitor="val_loss", save_best_only=True, verbose=1),
    ReduceLROnPlateau(monitor="val_loss", factor=0.5, patience=4, min_lr=1e-6, verbose=1),
]

# -----------------------
# TRAIN
# -----------------------
history = model.fit(
    train_inputs,
    y_tr,
    validation_data=(val_inputs, y_val),
    epochs=EPOCHS,
    batch_size=BATCH_SIZE,
    callbacks=callbacks,
    verbose=1
)

# Save final model (best saved by checkpoint)
model.save(os.path.join(OUT_DIR, "dl_car_price_model.h5"))
print("Saved DL model and artifacts to:", OUT_DIR)


Loaded: (19237, 23)
Categorical columns: ['Manufacturer', 'Model', 'Category', 'Fuel type', 'Gear box type', 'Drive wheels', 'Doors', 'Wheel', 'Color']
Numeric columns: ['Levy', 'Prod. year', 'Leather interior', 'Engine volume', 'Mileage', 'Cylinders', 'Airbags', 'Accident_history', 'Brand_value_index', 'Safety_features', 'Comfort_features', 'Modification_penalty']


Model: "functional_6"

┏━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┓
┃ Layer (type)        ┃ Output Shape      ┃    Param # ┃ Connected to      ┃
┡━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━┩
│ in_Manufacturer     │ (None, 1)         │          0 │ -                 │
│ (InputLayer)        │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ in_Model            │ (None, 1)         │          0 │ -                 │
│ (InputLayer)        │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ in_Category         │ (None, 1)         │          0 │ -                 │
│ (InputLayer)        │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ in_Fuel_type        │ (None, 1)         │          0 │ -                 │
│ (InputLayer)        │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ in_Gear_box_type    │ (None, 1)         │          0 │ -                 │
│ (InputLayer)        │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ in_Drive_wheels     │ (None, 1)         │          0 │ -                 │
│ (InputLayer)        │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ in_Doors            │ (None, 1)         │          0 │ -                 │
│ (InputLayer)        │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ in_Wheel            │ (None, 1)         │          0 │ -                 │
│ (InputLayer)        │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ in_Color            │ (None, 1)         │          0 │ -                 │
│ (InputLayer)        │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ emb_Manufacturer    │ (None, 1, 12)     │        792 │ in_Manufacturer[… │
│ (Embedding)         │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ emb_Model           │ (None, 1, 50)     │     79,550 │ in_Model[0][0]    │
│ (Embedding)         │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ emb_Category        │ (None, 1, 5)      │         60 │ in_Category[0][0] │
│ (Embedding)         │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ emb_Fuel_type       │ (None, 1, 4)      │         32 │ in_Fuel_type[0][… │
│ (Embedding)         │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ emb_Gear_box_type   │ (None, 1, 4)      │         20 │ in_Gear_box_type… │
│ (Embedding)         │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ emb_Drive_wheels    │ (None, 1, 4)      │         16 │ in_Drive_wheels[… │
│ (Embedding)         │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ emb_Doors           │ (None, 1, 4)      │         16 │ in_Doors[0][0]    │
│ (Embedding)         │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ emb_Wheel           │ (None, 1, 4)      │         12 │ in_Wheel[0][0]  

 Total params: 186,653 (729.11 KB)

 Trainable params: 186,653 (729.11 KB)

...

241/241 ━━━━━━━━━━━━━━━━━━━━ 9s 38ms/step - loss: 2.3883 - mae: 1.1778 - val_loss: 1.8334 - val_mae: 0.9601 - learning_rate: 5.0000e-04
Epoch 11/50
241/241 ━━━━━━━━━━━━━━━━━━━━ 0s 31ms/step - loss: 2.3154 - mae: 1.1614
Epoch 11: val_loss improved from 1.83341 to 1.82924, saving model to model\dl_car_price_model_best.h5


241/241 ━━━━━━━━━━━━━━━━━━━━ 9s 38ms/step - loss: 2.3507 - mae: 1.1715 - val_loss: 1.8292 - val_mae: 0.9581 - learning_rate: 5.0000e-04
Epoch 12/50
241/241 ━━━━━━━━━━━━━━━━━━━━ 0s 34ms/step - loss: 2.3076 - mae: 1.1560
Epoch 12: val_loss did not improve from 1.82924
241/241 ━━━━━━━━━━━━━━━━━━━━ 9s 39ms/step - loss: 2.3643 - mae: 1.1693 - val_loss: 1.8564 - val_mae: 0.9798 - learning_rate: 5.0000e-04
Epoch 13/50
241/241 ━━━━━━━━━━━━━━━━━━━━ 0s 38ms/step - loss: 2.3082 - mae: 1.1512
Epoch 13: val_loss did not improve from 1.82924
241/241 ━━━━━━━━━━━━━━━━━━━━ 11s 44ms/step - loss: 2.3199 - mae: 1.1550 - val_loss: 1.8551 - val_mae: 0.9816 - learning_rate: 5.0000e-04
Epoch 14/50
241/241 ━━━━━━━━━━━━━━━━━━━━ 0s 36ms/step - loss: 2.2675 - mae: 1.1401
Epoch 14: val_loss did not improve from 1.82924
241/241 ━━━━━━━━━━━━━━━━━━━━ 10s 41ms/step - loss: 2.2820 - mae: 1.1455 - val_loss: 1.8515 - val_mae: 0.9767 - learning_rate: 5.0000e-04
Epoch 15/50
240/241 ━━━━━━━━━━━━━━━━━━━━ 0s 32ms/step - loss:

241/241 ━━━━━━━━━━━━━━━━━━━━ 9s 39ms/step - loss: 2.3058 - mae: 1.1522 - val_loss: 1.8190 - val_mae: 0.9510 - learning_rate: 5.0000e-04
Epoch 16/50
240/241 ━━━━━━━━━━━━━━━━━━━━ 0s 30ms/step - loss: 2.2854 - mae: 1.1369
Epoch 16: val_loss did not improve from 1.81904
241/241 ━━━━━━━━━━━━━━━━━━━━ 9s 35ms/step - loss: 2.2837 - mae: 1.1417 - val_loss: 1.8356 - val_mae: 0.9814 - learning_rate: 5.0000e-04
Epoch 17/50
241/241 ━━━━━━━━━━━━━━━━━━━━ 0s 30ms/step - loss: 2.2401 - mae: 1.1330
Epoch 17: val_loss did not improve from 1.81904
241/241 ━━━━━━━━━━━━━━━━━━━━ 8s 35ms/step - loss: 2.2416 - mae: 1.1363 - val_loss: 1.8554 - val_mae: 0.9854 - learning_rate: 5.0000e-04
Epoch 18/50
241/241 ━━━━━━━━━━━━━━━━━━━━ 0s 32ms/step - loss: 2.2278 - mae: 1.1301
Epoch 18: val_loss did not improve from 1.81904
241/241 ━━━━━━━━━━━━━━━━━━━━ 9s 37ms/step - loss: 2.2248 - mae: 1.1298 - val_loss: 1.8313 - val_mae: 0.9798 - learning_rate: 5.0000e-04
Epoch 19/50
241/241 ━━━━━━━━━━━━━━━━━━━━ 0s 32ms/step - loss: 2

241/241 ━━━━━━━━━━━━━━━━━━━━ 10s 39ms/step - loss: 2.2468 - mae: 1.1283 - val_loss: 1.8120 - val_mae: 0.9497 - learning_rate: 2.5000e-04
Epoch 21/50
241/241 ━━━━━━━━━━━━━━━━━━━━ 0s 32ms/step - loss: 2.2328 - mae: 1.1250
Epoch 21: val_loss did not improve from 1.81201
241/241 ━━━━━━━━━━━━━━━━━━━━ 9s 37ms/step - loss: 2.2092 - mae: 1.1203 - val_loss: 1.8201 - val_mae: 0.9404 - learning_rate: 2.5000e-04
Epoch 22/50
240/241 ━━━━━━━━━━━━━━━━━━━━ 0s 32ms/step - loss: 2.2735 - mae: 1.1235
Epoch 22: val_loss did not improve from 1.81201
241/241 ━━━━━━━━━━━━━━━━━━━━ 9s 37ms/step - loss: 2.2255 - mae: 1.1219 - val_loss: 1.8159 - val_mae: 0.9635 - learning_rate: 2.5000e-04
Epoch 23/50
241/241 ━━━━━━━━━━━━━━━━━━━━ 0s 34ms/step - loss: 2.2017 - mae: 1.1158
Epoch 23: val_loss did not improve from 1.81201
241/241 ━━━━━━━━━━━━━━━━━━━━ 9s 39ms/step - loss: 2.1930 - mae: 1.1177 - val_loss: 1.8258 - val_mae: 0.9680 - learning_rate: 2.5000e-04
Epoch 24/50
241/241 ━━━━━━━━━━━━━━━━━━━━ 0s 33ms/step - loss: 

Saved DL model and artifacts to: model


In [64]:
# retrain_dl.py
# Improved CNN + BiLSTM with embeddings retraining script
# Expects train.csv at the same path: "./data/train.csv" or change DATA_PATH.

import os, re, pickle, time
import numpy as np, pandas as pd
from sklearn.preprocessing import LabelEncoder, StandardScaler
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

import tensorflow as tf
from tensorflow.keras.layers import Input, Embedding, Dense, Dropout, Reshape, Conv1D, MaxPooling1D, Bidirectional, LSTM, TimeDistributed, Concatenate
from tensorflow.keras.models import Model
from tensorflow.keras.callbacks import EarlyStopping, ModelCheckpoint, ReduceLROnPlateau
from tensorflow.keras.optimizers import Adam

# ---------- CONFIG ----------
DATA_PATH = r"C:\Users\Mohammed Yaseer\Desktop\CarPriceProject\data\train.csv"           # change if needed
OUT_DIR = "model_improved"
os.makedirs(OUT_DIR, exist_ok=True)

RANDOM_STATE = 42
EPOCHS = 60
BATCH_SIZE = 64

# ---------- UTIL ----------
def to_numeric_from_string(s):
    if pd.isna(s): return np.nan
    s = str(s).strip()
    s = re.sub(r'[^\d\.]', '', s)
    return float(s) if s != '' else np.nan

# ---------- LOAD ----------
df = pd.read_csv(DATA_PATH)
print("Loaded:", df.shape)

# ---------- CLEAN ----------
for col in ["Levy", "Engine volume", "Mileage"]:
    if col in df.columns:
        df[col] = df[col].apply(to_numeric_from_string)

for col in ["Leather interior", "Accident_history"]:
    if col in df.columns:
        df[col] = df[col].astype(str).str.lower().map({"yes":1,"no":0,"true":1,"false":0})
        df[col] = df[col].fillna(0).astype(int)

if "Prod. year" in df.columns:
    df["Prod. year"] = pd.to_numeric(df["Prod. year"], errors="coerce").fillna(df["Prod. year"].median())

if "ID" in df.columns:
    df = df.drop("ID", axis=1)

# ---------- TARGET: cap outliers then log1p ----------
y = df["Price"].astype(float)
cap_val = y.quantile(0.995)
print("Capping Price at:", cap_val)
y = np.minimum(y, cap_val)
y_log = np.log1p(y)

X = df.drop("Price", axis=1)

# ---------- FEATURE TYPES ----------
cat_cols = X.select_dtypes(include=['object']).columns.tolist()
num_cols = X.select_dtypes(include=[np.number]).columns.tolist()

print("Categorical columns:", cat_cols)
print("Numeric columns:", num_cols)

# ---------- ENCODERS ----------
encoders = {}
vocab_sizes = {}
cat_encoded = {}
for col in cat_cols:
    le = LabelEncoder()
    vals = X[col].fillna("___NA___").astype(str)
    le.fit(vals)
    encoders[col] = le
    cat_encoded[col] = le.transform(vals)
    vocab_sizes[col] = len(le.classes_) + 1

X_enc = X.copy()
for col in cat_cols:
    X_enc[col] = cat_encoded[col]

# ---------- SCALE NUMERIC ----------
scaler = StandardScaler()
X_enc[num_cols] = scaler.fit_transform(X_enc[num_cols])

# ---------- SAVE ENCODERS / SCALER / METADATA ----------
with open(os.path.join(OUT_DIR, "encoders.pkl"), "wb") as f:
    pickle.dump(encoders, f)
with open(os.path.join(OUT_DIR, "scaler.pkl"), "wb") as f:
    pickle.dump(scaler, f)
metadata = {"cat_cols": cat_cols, "num_cols": num_cols, "vocab_sizes": vocab_sizes, "log_target": True}
with open(os.path.join(OUT_DIR, "metadata.pkl"), "wb") as f:
    pickle.dump(metadata, f)

# ---------- BUILD MODEL ----------
cat_inputs = []
embeddings = []
for col in cat_cols:
    vocab = vocab_sizes[col]
    emb_dim = min(64, max(8, int(1.6 * np.sqrt(vocab))))
    safe = col.replace(" ", "_")
    inp = Input(shape=(1,), name=f"in_{safe}")
    emb = Embedding(input_dim=vocab, output_dim=emb_dim, name=f"emb_{safe}")(inp)
    emb = Reshape((emb_dim,))(emb)
    cat_inputs.append(inp)
    embeddings.append(emb)

num_input = Input(shape=(len(num_cols),), name="numeric_input")
num_proj = Dense(64, activation="relu")(num_input)

embed_dims = [int(e.shape[-1]) for e in embeddings]
COMMON_DIM = max(64, max(embed_dims))

projected_tokens = []
for emb in embeddings:
    p = Dense(COMMON_DIM, activation="relu")(emb)
    token = Reshape((1, COMMON_DIM))(p)
    projected_tokens.append(token)

num_p = Dense(COMMON_DIM, activation="relu")(num_proj)
projected_tokens.append(Reshape((1, COMMON_DIM))(num_p))

sequence = Concatenate(axis=1)(projected_tokens)
sequence_proj = TimeDistributed(Dense(COMMON_DIM, activation="relu"))(sequence)

x = Conv1D(filters=128, kernel_size=2, activation="relu", padding="same")(sequence_proj)
x = MaxPooling1D(pool_size=1)(x)
x = Bidirectional(LSTM(128, return_sequences=False))(x)
x = Dropout(0.35)(x)
x = Dense(256, activation="relu")(x)
x = Dropout(0.3)(x)
x = Dense(128, activation="relu")(x)
x = Dropout(0.2)(x)
out = Dense(1, activation="linear")(x)

model = Model(inputs=cat_inputs + [num_input], outputs=out)
model.compile(optimizer=Adam(learning_rate=1e-3), loss="mse", metrics=["mae"])
model.summary()

# ---------- PREPARE ARRAYS ----------
model_inputs = {}
for col in cat_cols:
    safe = col.replace(" ", "_")
    model_inputs[f"in_{safe}"] = X_enc[col].astype("int32").values.reshape(-1,1)
model_inputs["numeric_input"] = X_enc[num_cols].astype("float32").values

# train/val split
train_idx, val_idx = train_test_split(np.arange(X_enc.shape[0]), test_size=0.2, random_state=RANDOM_STATE)
train_inputs = {k: v[train_idx] for k,v in model_inputs.items()}
val_inputs = {k: v[val_idx] for k,v in model_inputs.items()}
y_tr = y_log[train_idx]
y_val = y_log[val_idx]

# callbacks
ckpt = os.path.join(OUT_DIR, "dl_car_price_model_best.keras")
callbacks = [
    EarlyStopping(monitor="val_loss", patience=8, restore_best_weights=True),
    ModelCheckpoint(ckpt, monitor="val_loss", save_best_only=True, verbose=1),
    ReduceLROnPlateau(monitor="val_loss", factor=0.5, patience=4, min_lr=1e-6, verbose=1)
]

# ---------- TRAIN ----------
start = time.time()
history = model.fit(train_inputs, y_tr, validation_data=(val_inputs, y_val),
                    epochs=EPOCHS, batch_size=BATCH_SIZE, callbacks=callbacks, verbose=1)
end = time.time()
print("Training time (s):", end-start)

# ---------- SAVE ----------
model.save(os.path.join(OUT_DIR, "dl_car_price_model_final.keras"))
model.save(os.path.join(OUT_DIR, "dl_car_price_model_final.h5"), include_optimizer=False)
with open(os.path.join(OUT_DIR, "training_history.pkl"), "wb") as f:
    pickle.dump(history.history, f)

# ---------- EVAL ----------
pred_val = model.predict(val_inputs).reshape(-1)
pred_val_price = np.expm1(pred_val)
y_val_price = np.expm1(y_val)
mae = mean_absolute_error(y_val_price, pred_val_price)
rmse = np.sqrt(mean_squared_error(y_val_price, pred_val_price))
r2 = r2_score(y_val_price, pred_val_price)
print(f"Val MAE: {mae:.2f}, RMSE: {rmse:.2f}, R2: {r2:.4f}")

# save some sample predictions
sample_df = X.iloc[val_idx][:20].copy()
sample_df["true_price"] = np.expm1(y_val)[:20]
sample_df["pred_price"] = pred_val_price[:20]
sample_df.to_csv(os.path.join(OUT_DIR, "sample_predictions.csv"), index=False)

print("Artifacts saved to:", OUT_DIR)


Loaded: (19237, 23)
Capping Price at: 103491.0
Categorical columns: ['Manufacturer', 'Model', 'Category', 'Fuel type', 'Gear box type', 'Drive wheels', 'Doors', 'Wheel', 'Color']
Numeric columns: ['Levy', 'Prod. year', 'Leather interior', 'Engine volume', 'Mileage', 'Cylinders', 'Airbags', 'Accident_history', 'Brand_value_index', 'Safety_features', 'Comfort_features', 'Modification_penalty']


Model: "functional_8"

┏━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┓
┃ Layer (type)        ┃ Output Shape      ┃    Param # ┃ Connected to      ┃
┡━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━┩
│ in_Manufacturer     │ (None, 1)         │          0 │ -                 │
│ (InputLayer)        │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ in_Model            │ (None, 1)         │          0 │ -                 │
│ (InputLayer)        │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ in_Category         │ (None, 1)         │          0 │ -                 │
│ (InputLayer)        │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ in_Fuel_type        │ (None, 1)         │          0 │ -                 │
│ (InputLayer)        │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ in_Gear_box_type    │ (None, 1)         │          0 │ -                 │
│ (InputLayer)        │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ in_Drive_wheels     │ (None, 1)         │          0 │ -                 │
│ (InputLayer)        │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ in_Doors            │ (None, 1)         │          0 │ -                 │
│ (InputLayer)        │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ in_Wheel            │ (None, 1)         │          0 │ -                 │
│ (InputLayer)        │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ in_Color            │ (None, 1)         │          0 │ -                 │
│ (InputLayer)        │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ emb_Manufacturer    │ (None, 1, 12)     │        792 │ in_Manufacturer[… │
│ (Embedding)         │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ emb_Model           │ (None, 1, 63)     │    100,233 │ in_Model[0][0]    │
│ (Embedding)         │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ emb_Category        │ (None, 1, 8)      │         96 │ in_Category[0][0] │
│ (Embedding)         │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ emb_Fuel_type       │ (None, 1, 8)      │         64 │ in_Fuel_type[0][… │
│ (Embedding)         │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ emb_Gear_box_type   │ (None, 1, 8)      │         40 │ in_Gear_box_type… │
│ (Embedding)         │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ emb_Drive_wheels    │ (None, 1, 8)      │         32 │ in_Drive_wheels[… │
│ (Embedding)         │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ emb_Doors           │ (None, 1, 8)      │         32 │ in_Doors[0][0]    │
│ (Embedding)         │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ emb_Wheel           │ (None, 1, 8)      │         24 │ in_Wheel[0][0]  

 Total params: 498,058 (1.90 MB)

 Trainable params: 498,058 (1.90 MB)

 Non-trainable params: 0 (0.00 B)

Epoch 1/60
241/241 ━━━━━━━━━━━━━━━━━━━━ 0s 33ms/step - loss: 16.5554 - mae: 2.8479
Epoch 1: val_loss improved from None to 1.92596, saving model to model_improved\dl_car_price_model_best.keras
241/241 ━━━━━━━━━━━━━━━━━━━━ 32s 47ms/step - loss: 6.5157 - mae: 1.7734 - val_loss: 1.9260 - val_mae: 1.0416 - learning_rate: 0.0010
Epoch 2/60
241/241 ━━━━━━━━━━━━━━━━━━━━ 0s 31ms/step - loss: 2.7667 - mae: 1.2910
Epoch 2: val_loss improved from 1.92596 to 1.88714, saving model to model_improved\dl_car_price_model_best.keras
241/241 ━━━━━━━━━━━━━━━━━━━━ 9s 38ms/step - loss: 2.6885 - mae: 1.2700 - val_loss: 1.8871 - val_mae: 0.9556 - learning_rate: 0.0010
Epoch 3/60
240/241 ━━━━━━━━━━━━━━━━━━━━ 0s 29ms/step - loss: 2.6222 - mae: 1.2494
Epoch 3: val_loss did not improve from 1.88714
241/241 ━━━━━━━━━━━━━━━━━━━━ 8s 33ms/step - loss: 2.5727 - mae: 1.2366 - val_loss: 2.2550 - val_mae: 1.0395 - learning_rate: 0.0010
Epoch 4/60
241/241 ━━━━━━━━━━━━━━━━━━━━ 0s 32ms/step - loss: 2.5584 - mae: 1.2314
Epoc

121/121 ━━━━━━━━━━━━━━━━━━━━ 4s 25ms/step
Val MAE: 8934.70, RMSE: 15006.28, R2: 0.1763
Artifacts saved to: model_improved


In [70]:
# retrain_tabular_dense.py
# Embeddings + Dense "tabular" model for car price regression
# Save artifacts to ./model_tabular
import os, re, time, pickle
import numpy as np
import pandas as pd
from sklearn.preprocessing import LabelEncoder, StandardScaler
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

import tensorflow as tf
from tensorflow.keras.layers import Input, Embedding, Dense, Dropout, Flatten, Concatenate, BatchNormalization
from tensorflow.keras.models import Model
from tensorflow.keras.callbacks import EarlyStopping, ModelCheckpoint, ReduceLROnPlateau
from tensorflow.keras.optimizers import Adam

# ---------- CONFIG ----------
DATA_PATH = r"C:\Users\Mohammed Yaseer\Desktop\CarPriceProject\data\train.csv"     # change if your file is elsewhere
OUT_DIR = "model_tabular"
os.makedirs(OUT_DIR, exist_ok=True)

RANDOM_STATE = 42
EPOCHS = 80
BATCH_SIZE = 128
LEARNING_RATE = 1e-3
VALIDATION_SPLIT = 0.2

# ---------- HELPERS ----------
def to_numeric_from_string(s):
    if pd.isna(s): return np.nan
    s = str(s).strip()
    s = re.sub(r'[^\d\.]', '', s)
    return float(s) if s != '' else np.nan

# ---------- LOAD ----------
df = pd.read_csv(DATA_PATH)
print("Loaded:", df.shape)

# ---------- BASIC CLEAN ----------
# convert numeric-like strings
for col in ["Levy","Engine volume","Mileage"]:
    if col in df.columns:
        df[col] = df[col].apply(to_numeric_from_string).fillna(0)

# binary fields
for col in ["Leather interior","Accident_history"]:
    if col in df.columns:
        df[col] = df[col].astype(str).str.lower().map({"yes":1,"no":0,"true":1,"false":0}).fillna(0).astype(int)

# Prod. year fallback
if "Prod. year" in df.columns:
    df["Prod. year"] = pd.to_numeric(df["Prod. year"], errors="coerce").fillna(df["Prod. year"].median())

# remove ID if present
if "ID" in df.columns:
    df = df.drop("ID", axis=1)

# ---------- TARGET PREP ----------
y_raw = df["Price"].astype(float)
cap_val = y_raw.quantile(0.995)            # winsorize tail
y_capped = np.minimum(y_raw, cap_val)
y = np.log1p(y_capped)                    # log1p transform target

X = df.drop("Price", axis=1)

# ---------- FEATURES ----------
cat_cols = X.select_dtypes(include=['object']).columns.tolist()
num_cols = X.select_dtypes(include=[np.number]).columns.tolist()

print("Categorical columns:", cat_cols)
print("Numeric columns:", num_cols)

# ---------- ENCODING ----------
encoders = {}
vocab_sizes = {}
X_enc = X.copy()

for col in cat_cols:
    le = LabelEncoder()
    vals = X[col].fillna("___NA___").astype(str)
    le.fit(vals)
    encoders[col] = le
    X_enc[col] = le.transform(vals).astype("int32")
    # +1 for safety/padding (not used), keep dims sensible
    vocab_sizes[col] = len(le.classes_) + 1

# ---------- SCALE NUMERIC ----------
scaler = StandardScaler()
X_enc[num_cols] = scaler.fit_transform(X_enc[num_cols].astype(float))

# ---------- SAVE PREPROCESSORS ----------
with open(os.path.join(OUT_DIR, "encoders.pkl"), "wb") as f:
    pickle.dump(encoders, f)
with open(os.path.join(OUT_DIR, "scaler.pkl"), "wb") as f:
    pickle.dump(scaler, f)
metadata = {"cat_cols": cat_cols, "num_cols": num_cols, "vocab_sizes": vocab_sizes, "log_target": True}
with open(os.path.join(OUT_DIR, "metadata.pkl"), "wb") as f:
    pickle.dump(metadata, f)

# ---------- BUILD TABULAR MODEL ----------
# Inputs for categorical embeddings
cat_inputs = []
embeddings = []
for col in cat_cols:
    vocab = vocab_sizes[col]
    # simple heuristic for embedding dim
    emb_dim = min(50, max(4, int(np.ceil(np.sqrt(vocab)))))
    inp = Input(shape=(1,), name=f"in_{col.replace(' ','_')}")
    emb = Embedding(input_dim=vocab, output_dim=emb_dim, name=f"emb_{col.replace(' ','_')}", embeddings_regularizer=None)(inp)
    flat = Flatten()(emb)
    cat_inputs.append(inp)
    embeddings.append(flat)

# Numeric input
num_input = Input(shape=(len(num_cols),), name="numeric_input")

# Concatenate embeddings + numeric
x = Concatenate()(embeddings + [num_input])

# Fully connected head (wide & deep)
x = Dense(512, activation="relu")(x)
x = BatchNormalization()(x)
x = Dropout(0.35)(x)

x = Dense(256, activation="relu")(x)
x = BatchNormalization()(x)
x = Dropout(0.25)(x)

x = Dense(128, activation="relu")(x)
x = BatchNormalization()(x)
x = Dropout(0.2)(x)

out = Dense(1, activation="linear", name="price_out")(x)

model = Model(inputs=cat_inputs + [num_input], outputs=out)
model.compile(optimizer=Adam(learning_rate=LEARNING_RATE), loss="mse", metrics=["mae"])
model.summary()

# ---------- PREPARE ARRAYS ----------
model_inputs = {}
for col in cat_cols:
    safe = col.replace(" ", "_")
    model_inputs[f"in_{safe}"] = X_enc[col].astype("int32").values.reshape(-1,1)
model_inputs["numeric_input"] = X_enc[num_cols].astype("float32").values

# ---------- SPLIT ----------
idx = np.arange(X_enc.shape[0])
train_idx, val_idx = train_test_split(idx, test_size=VALIDATION_SPLIT, random_state=RANDOM_STATE)

train_inputs = {k: v[train_idx] for k,v in model_inputs.items()}
val_inputs = {k: v[val_idx] for k,v in model_inputs.items()}
y_tr = y.values[train_idx]
y_val = y.values[val_idx]

# ---------- CALLBACKS ----------
ckpt = os.path.join(OUT_DIR, "dl_tabular_model_best.keras")
callbacks = [
    EarlyStopping(monitor="val_loss", patience=10, restore_best_weights=True, verbose=1),
    ModelCheckpoint(ckpt, monitor="val_loss", save_best_only=True, verbose=1),
    ReduceLROnPlateau(monitor="val_loss", factor=0.5, patience=5, min_lr=1e-6, verbose=1)
]

# ---------- TRAIN ----------
start = time.time()
history = model.fit(train_inputs, y_tr, validation_data=(val_inputs, y_val),
                    epochs=EPOCHS, batch_size=BATCH_SIZE, callbacks=callbacks, verbose=1)
print("Training time (s):", time.time() - start)

# ---------- SAVE ----------
model.save(os.path.join(OUT_DIR, "dl_tabular_model_final.keras"))
model.save(os.path.join(OUT_DIR, "dl_tabular_model_final.h5"), include_optimizer=False)
with open(os.path.join(OUT_DIR, "training_history.pkl"), "wb") as f:
    pickle.dump(history.history, f)

# ---------- EVAL ----------
pred_val_log = model.predict(val_inputs).reshape(-1)
pred_val_price = np.expm1(pred_val_log)
y_val_price = np.expm1(y_val)

mae = mean_absolute_error(y_val_price, pred_val_price)
rmse = np.sqrt(mean_squared_error(y_val_price, pred_val_price))
r2 = r2_score(y_val_price, pred_val_price)
print(f"Val MAE: {mae:.2f}, RMSE: {rmse:.2f}, R2: {r2:.4f}")

# sample predictions
sample_df = X.iloc[val_idx][:50].copy().reset_index(drop=True)
sample_df["true_price"] = y_val_price[:50]
sample_df["pred_price"] = pred_val_price[:50]
sample_df.to_csv(os.path.join(OUT_DIR, "sample_predictions.csv"), index=False)

print("Saved artifacts to", OUT_DIR)


Loaded: (19237, 18)
Categorical columns: ['Manufacturer', 'Model', 'Category', 'Fuel type', 'Gear box type', 'Drive wheels', 'Doors', 'Wheel', 'Color']
Numeric columns: ['Levy', 'Prod. year', 'Leather interior', 'Engine volume', 'Mileage', 'Cylinders', 'Airbags']


Model: "functional_12"

┏━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┓
┃ Layer (type)        ┃ Output Shape      ┃    Param # ┃ Connected to      ┃
┡━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━┩
│ in_Manufacturer     │ (None, 1)         │          0 │ -                 │
│ (InputLayer)        │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ in_Model            │ (None, 1)         │          0 │ -                 │
│ (InputLayer)        │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ in_Category         │ (None, 1)         │          0 │ -                 │
│ (InputLayer)        │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ in_Fuel_type        │ (None, 1)         │          0 │ -                 │
│ (InputLayer)        │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ in_Gear_box_type    │ (None, 1)         │          0 │ -                 │
│ (InputLayer)        │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ in_Drive_wheels     │ (None, 1)         │          0 │ -                 │
│ (InputLayer)        │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ in_Doors            │ (None, 1)         │          0 │ -                 │
│ (InputLayer)        │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ in_Wheel            │ (None, 1)         │          0 │ -                 │
│ (InputLayer)        │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ in_Color            │ (None, 1)         │          0 │ -                 │
│ (InputLayer)        │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ emb_Manufacturer    │ (None, 1, 9)      │        594 │ in_Manufacturer[… │
│ (Embedding)         │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ emb_Model           │ (None, 1, 40)     │     63,640 │ in_Model[0][0]    │
│ (Embedding)         │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ emb_Category        │ (None, 1, 4)      │         48 │ in_Category[0][0] │
│ (Embedding)         │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ emb_Fuel_type       │ (None, 1, 4)      │         32 │ in_Fuel_type[0][… │
│ (Embedding)         │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ emb_Gear_box_type   │ (None, 1, 4)      │         20 │ in_Gear_box_type… │
│ (Embedding)         │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ emb_Drive_wheels    │ (None, 1, 4)      │         16 │ in_Drive_wheels[… │
│ (Embedding)         │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ emb_Doors           │ (None, 1, 4)      │         16 │ in_Doors[0][0]    │
│ (Embedding)         │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ emb_Wheel           │ (None, 1, 4)      │         12 │ in_Wheel[0][0]  

 Total params: 276,432 (1.05 MB)

 Trainable params: 274,640 (1.05 MB)

 Non-trainable params: 1,792 (7.00 KB)

Epoch 1/80
118/121 ━━━━━━━━━━━━━━━━━━━━ 0s 19ms/step - loss: 153.1269 - mae: 12.2312
Epoch 1: val_loss improved from None to 87.40141, saving model to model_tabular\dl_tabular_model_best.keras
121/121 ━━━━━━━━━━━━━━━━━━━━ 22s 40ms/step - loss: 136.2415 - mae: 11.5002 - val_loss: 87.4014 - val_mae: 9.2285 - learning_rate: 0.0010
Epoch 2/80
120/121 ━━━━━━━━━━━━━━━━━━━━ 0s 21ms/step - loss: 66.8086 - mae: 7.7654
Epoch 2: val_loss improved from 87.40141 to 7.40739, saving model to model_tabular\dl_tabular_model_best.keras
121/121 ━━━━━━━━━━━━━━━━━━━━ 4s 30ms/step - loss: 44.7630 - mae: 6.0531 - val_loss: 7.4074 - val_mae: 2.4953 - learning_rate: 0.0010
Epoch 3/80
121/121 ━━━━━━━━━━━━━━━━━━━━ 0s 22ms/step - loss: 8.9458 - mae: 2.3472 
Epoch 3: val_loss improved from 7.40739 to 1.78227, saving model to model_tabular\dl_tabular_model_best.keras
121/121 ━━━━━━━━━━━━━━━━━━━━ 4s 30ms/step - loss: 7.2609 - mae: 2.0491 - val_loss: 1.7823 - val_mae: 0.8863 - learning_rate: 0.0010
Epoch 4/80
121/121

121/121 ━━━━━━━━━━━━━━━━━━━━ 3s 14ms/step   
Val MAE: 234689.10, RMSE: 442797.20, R2: 0.5836
Saved artifacts to model_tabular


In [67]:
import os, pickle, numpy as np, pandas as pd
from tensorflow.keras.models import load_model

MDIR = "model_tabular"
print("Exists:", os.path.exists(MDIR))
print("Files:", os.listdir(MDIR))

# load artifacts
with open(os.path.join(MDIR, "metadata.pkl"), "rb") as f:
    metadata = pickle.load(f)
with open(os.path.join(MDIR, "encoders.pkl"), "rb") as f:
    encoders = pickle.load(f)
with open(os.path.join(MDIR, "scaler.pkl"), "rb") as f:
    scaler = pickle.load(f)

print("\nmetadata keys:", metadata.keys())
cat_cols = metadata["cat_cols"]
num_cols = metadata["num_cols"]
print("cat_cols:", cat_cols)
print("num_cols:", num_cols)

# show scaler stats
print("\nSCALER MEAN & SCALE (first 12 numeric cols):")
for i,c in enumerate(num_cols[:12]):
    mean = scaler.mean_[i] if hasattr(scaler, "mean_") else None
    scale = scaler.scale_[i] if hasattr(scaler, "scale_") else None
    print(f"{i:02d}. {c:30s} mean={mean} scale={scale}")

# load model
model_file_candidates = [f for f in os.listdir(MDIR) if f.endswith(".keras") or f.endswith(".h5")]
print("\nCandidate model files:", model_file_candidates)
model = None
if "dl_tabular_model_best.keras" in model_file_candidates:
    model = load_model(os.path.join(MDIR, "dl_tabular_model_best.keras"), compile=False)
else:
    model = load_model(os.path.join(MDIR, model_file_candidates[0]), compile=False)
print("Loaded model:", model.name)
print(model.summary())

# Load sample predictions saved by training (if exists)
sample_csv = os.path.join(MDIR, "sample_predictions.csv")
if os.path.exists(sample_csv):
    s = pd.read_csv(sample_csv)
    print("\nSample predictions (first 10):")
    print(s.head(10).to_string(index=False))
else:
    print("\nNo sample_predictions.csv found")

# Quick sanity on target distribution from original train CSV (if available)
# The training script used DATA_PATH variable — try to find train.csv in repo
possible_paths = ["data/train.csv", "train.csv", "../data/train.csv", os.path.join(os.getcwd(),"data","train.csv")]
for p in possible_paths:
    if os.path.exists(p):
        print("\nFound train file at:", p)
        df = pd.read_csv(p)
        print("Train shape:", df.shape)
        if "Price" in df.columns:
            pr = df["Price"].astype(float)
            print("Price describe:")
            print(pr.describe().to_string())
            print("Price quantiles (0.001,0.01,0.05,0.5,0.95,0.99,0.995):")
            print(pr.quantile([0.001,0.01,0.05,0.5,0.95,0.99,0.995]).to_dict())
        break

# If we have validation artifacts, compute a baseline predicted mean:
try:
    # try to load validation inputs dumped during training (not present usually)
    pass
except Exception as e:
    print("Skipping val prediction check:", e)

print("\nDONE. Paste the above output here.")


Exists: True
Files: ['dl_tabular_model_best.keras', 'dl_tabular_model_final.h5', 'dl_tabular_model_final.keras', 'encoders.pkl', 'metadata.pkl', 'sample_predictions.csv', 'scaler.pkl', 'training_history.pkl']

metadata keys: dict_keys(['cat_cols', 'num_cols', 'vocab_sizes', 'log_target'])
cat_cols: ['Manufacturer', 'Model', 'Category', 'Fuel type', 'Gear box type', 'Drive wheels', 'Doors', 'Wheel', 'Color']
num_cols: ['Levy', 'Prod. year', 'Leather interior', 'Engine volume', 'Mileage', 'Cylinders', 'Airbags', 'Accident_history', 'Brand_value_index', 'Safety_features', 'Comfort_features', 'Modification_penalty']

SCALER MEAN & SCALE (first 12 numeric cols):
00. Levy                           mean=632.5286687113376 scale=567.7069321003172
01. Prod. year                     mean=2010.9128242449447 scale=5.668525654712376
02. Leather interior               mean=0.7253729791547538 scale=0.44632613666119864
03. Engine volume                  mean=2.307989811301139 scale=0.8777816926806773
0

Model: "functional_9"

┏━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┓
┃ Layer (type)        ┃ Output Shape      ┃    Param # ┃ Connected to      ┃
┡━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━┩
│ in_Manufacturer     │ (None, 1)         │          0 │ -                 │
│ (InputLayer)        │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ in_Model            │ (None, 1)         │          0 │ -                 │
│ (InputLayer)        │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ in_Category         │ (None, 1)         │          0 │ -                 │
│ (InputLayer)        │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ in_Fuel_type        │ (None, 1)         │          0 │ -                 │
│ (InputLayer)        │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ in_Gear_box_type    │ (None, 1)         │          0 │ -                 │
│ (InputLayer)        │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ in_Drive_wheels     │ (None, 1)         │          0 │ -                 │
│ (InputLayer)        │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ in_Doors            │ (None, 1)         │          0 │ -                 │
│ (InputLayer)        │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ in_Wheel            │ (None, 1)         │          0 │ -                 │
│ (InputLayer)        │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ in_Color            │ (None, 1)         │          0 │ -                 │
│ (InputLayer)        │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ emb_Manufacturer    │ (None, 1, 9)      │        594 │ in_Manufacturer[… │
│ (Embedding)         │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ emb_Model           │ (None, 1, 40)     │     63,640 │ in_Model[0][0]    │
│ (Embedding)         │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ emb_Category        │ (None, 1, 4)      │         48 │ in_Category[0][0] │
│ (Embedding)         │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ emb_Fuel_type       │ (None, 1, 4)      │         32 │ in_Fuel_type[0][… │
│ (Embedding)         │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ emb_Gear_box_type   │ (None, 1, 4)      │         20 │ in_Gear_box_type… │
│ (Embedding)         │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ emb_Drive_wheels    │ (None, 1, 4)      │         16 │ in_Drive_wheels[… │
│ (Embedding)         │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ emb_Doors           │ (None, 1, 4)      │         16 │ in_Doors[0][0]    │
│ (Embedding)         │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ emb_Wheel           │ (None, 1, 4)      │         12 │ in_Wheel[0][0]  

 Total params: 278,992 (1.06 MB)

 Trainable params: 277,200 (1.06 MB)

 Non-trainable params: 1,792 (7.00 KB)

None

Sample predictions (first 10):
  Levy  Manufacturer    Model  Prod. year  Category  Leather interior      Fuel type  Engine volume  Mileage  Cylinders Gear box type Drive wheels Doors            Wheel  Color  Airbags  Accident_history  Brand_value_index  Safety_features  Comfort_features  Modification_penalty  true_price  pred_price
 259.0     CHEVROLET     Volt        2014 Hatchback                 0 Plug-in Hybrid            1.4  65000.0          4     Automatic        Front 4-May       Left wheel Silver       10                 0                  4                4                 8                     1     27284.0  20684.0450
   0.0 MERCEDES-BENZ Sprinter        1997  Microbus                 1         Diesel            2.9   3333.0          6        Manual         Rear 2-Mar       Left wheel    Red        2                 0                  4                4                 5                     1     10349.0  11022.5780
   0.0 MERCEDES-BENZ    C 180        1996     Sedan